In [1]:
import datetime
import gc  # for memory management (delete temporary variables + garbage collect)
import pickle  # allows saving python objects (lists, tuples)
import re
from pathlib import Path

import geopandas  # for geocoding tabular dataset
import matplotlib.pyplot as plt  # for plotting
import numpy as np  # numpy for fast vectorized computations (matrix multiplications)
import pandas as pd  # for working with tabular datasets
from tqdm import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

data_dir = Path.cwd().parent / "data"

global_epsg = "epsg:4326"
local_epsg = "epsg:6583"

PRE_COVID = True  # Set to False for post-COVID data (March 23, 2020 onwards), by default, biz - pre_covid

gc.collect()

0

#### Merged census dataset

In [ ]:
census_merged = geopandas.read_file(
    data_dir / "census/census_merged_fine.gpkg"
)  # biz paper

#### Filtering based on Dallas city boundary

In [5]:
def append_block_id(dataset, shp, points, column_block):
    """putting geometric points into census blocks (geometric polygons)

    Args:
        dataset (pd.DataFrame):
        shp (pd.DataFrame): shpaefile of the blocks
        column_point (str): column name containing the Points
        column_block (str): column name containing the Polygons

    Returns:
        pd.DataFrame: dataset with census ids appended
    """

    print(f"Putting {len(dataset)} points into {len(shp)} blocks...")
    num_blocks = len(shp)
    dataset[column_block] = -1  # setinel value

    for i in tqdm(range(num_blocks)):  # loop over census blocks
        indices_within = points.within(
            shp.geometry.iloc[i]
        )  # check if points is in blocks
        if indices_within.any():
            dataset.loc[indices_within, column_block] = i

    return dataset

In [7]:
# Define field selections for data loading
offenders_fields = [
    "incident_num",
    "Race",
    "Sex",
    "Ar_LAT",
    "Ar_LON",
    "H_LAT",
    "H_LON",
]

incidents_fields = [
    "incident_num",
    "Type of Incident",
    "Type  Location",
    "Date1 of Occurrence",
    "Victim Type",
    "Victim Race",
    "Victim Gender",
    "NIBRS Crime Category",
    "V_LAT",
    "V_LON",
    "I_LAT",
    "I_LON",
]

# Define data paths using pathlib
offenders_path = data_dir / "csv_files/offenders_geocoded.csv"
incidents_path = data_dir / "csv_files/incidents_geocoded.csv"

# Load data
offenders = pd.read_csv(offenders_path, usecols=offenders_fields)
incidents = pd.read_csv(incidents_path, usecols=incidents_fields)

print(f"Loaded {len(incidents)} incidents and {len(offenders)} offenders")

# Define all coordinate mappings
coordinate_mappings = [
    (offenders, "H_LON", "H_LAT", "H_CensusID"),
    (incidents, "V_LON", "V_LAT", "V_CensusID"),
    (incidents, "I_LON", "I_LAT", "I_CensusID"),
]

# Process all coordinate mappings
for df, lon_col, lat_col, output_col in coordinate_mappings:
    points = geopandas.GeoSeries.from_xy(df[lon_col], df[lat_col], crs=global_epsg)
    append_block_id(df, census_merged, points, output_col)

# Merge datasets
merged = incidents.merge(offenders, on="incident_num", how="outer")
print(f"Merged dataset has {len(merged)} records")

Loaded 839544 incidents and 92136 offenders
Putting 92136 points into 22564 blocks...


100%|██████████| 22564/22564 [00:42<00:00, 536.23it/s]


Putting 839544 points into 22564 blocks...


100%|██████████| 22564/22564 [06:16<00:00, 59.96it/s]


Putting 839544 points into 22564 blocks...


100%|██████████| 22564/22564 [06:15<00:00, 60.06it/s]


Merged dataset has 869024 records


#### Filtering based on date range

In [8]:
if PRE_COVID:
    # Pre-COVID: June 1, 2014 to March 23, 2020
    start_date = datetime.datetime(2014, 6, 1)
    end_date = datetime.datetime(2020, 3, 23)
else:
    # Post-COVID: March 23, 2020 onwards (no end date)
    start_date = datetime.datetime(2020, 3, 23)
    end_date = None

# Convert and filter dates
merged["Date1 of Occurrence"] = pd.to_datetime(merged["Date1 of Occurrence"])

if end_date is not None:
    # Both start and end date filtering
    date_mask = (merged["Date1 of Occurrence"] >= start_date) & (
        merged["Date1 of Occurrence"] <= end_date
    )
else:
    # Only start date filtering (for post-COVID)
    date_mask = merged["Date1 of Occurrence"] >= start_date

merged = merged.loc[date_mask].copy()

# Get actual date range and convert to days since start
actual_min_date = merged["Date1 of Occurrence"].min()
actual_max_date = merged["Date1 of Occurrence"].max()
date_range = actual_max_date - actual_min_date

print(
    f"Date range: {actual_min_date.date()} to {actual_max_date.date()} ({date_range.days} days)"
)
print(f"Filtered dataset has {len(merged)} records")

# Convert dates to days since start for modeling
merged["Date1 of Occurrence"] = (
    merged["Date1 of Occurrence"] - actual_min_date
).dt.days.astype(int)

# Clean up memory
del incidents, offenders
gc.collect()

Date range: 2014-06-01 to 2020-03-23 (2122 days)
Filtered dataset has 656920 records


21

#### Filtering based on traveled distance to crime

In [9]:
def filter_by_distance(
    df, lon_col, lat_col, i_lon_col, i_lat_col, min_distance_meters=10
):
    # Check for valid coordinates (not null)
    valid_coords = (
        (~df[lon_col].isna())
        & (~df[lat_col].isna())
        & (~df[i_lon_col].isna())
        & (~df[i_lat_col].isna())
    )

    if not valid_coords.any():
        return np.full(len(df), False)

    # Create geographic points for distance calculations
    location_points = geopandas.GeoSeries.from_xy(
        x=df[lon_col][valid_coords], y=df[lat_col][valid_coords], crs=global_epsg
    ).to_crs(local_epsg)

    incident_points = geopandas.GeoSeries.from_xy(
        x=df[i_lon_col][valid_coords], y=df[i_lat_col][valid_coords], crs=global_epsg
    ).to_crs(local_epsg)

    # Calculate distances
    distances = location_points.distance(incident_points, align=False)

    # Get indices that meet minimum distance requirement
    valid_distance_indices = np.arange(len(df))[valid_coords][
        distances > min_distance_meters
    ]

    # Create boolean mask
    valid_mask = np.full(len(df), False)
    valid_mask[valid_distance_indices] = True

    return valid_mask


MIN_DISTANCE_METERS = 10

# filters out incidents not inside Dallas
df = merged.loc[merged["I_CensusID"] != -1]

offenders_valid = filter_by_distance(
    df, "H_LON", "H_LAT", "I_LON", "I_LAT", MIN_DISTANCE_METERS
)
victims_valid = filter_by_distance(
    df, "V_LON", "V_LAT", "I_LON", "I_LAT", MIN_DISTANCE_METERS
)

print(f"Valid offenders (>10m from incident): {offenders_valid.sum()}")
print(f"Valid victims (>10m from incident): {victims_valid.sum()}")

# # Clean up memory
# del merged
# gc.collect()

# Create final filtered datasets
offenders = df[offenders_valid]
offenders = offenders.loc[offenders["H_CensusID"] != -1]

victims = df[victims_valid]
victims = victims.loc[victims["V_CensusID"] != -1]

print(f"Final offenders shape: {offenders.shape}")
print(f"Final victims shape: {victims.shape}")

Valid offenders (>10m from incident): 55294
Valid victims (>10m from incident): 431387
Final offenders shape: (39132, 21)
Final victims shape: (323883, 21)


#### Extract Labels

In [ ]:
from dcm.protocols import AgentFeatures

# Load race mapping
with open(data_dir / "mappers/race_remap.pkl", "rb") as f:
    race_remap = pickle.load(f)

crime_remap_df = pd.read_csv(data_dir / "mappers/recategorize_crime.csv")
crime_type_mapping = dict(
    zip(crime_remap_df["Type of Incident"], crime_remap_df["rNIBRS Crime Category"])
)

# Define crime types list (reuse from existing)
crime_types = [
    "burglary_breaking_entering",
    "motor_vehicle_theft",
    "larceny_theft_offenses",
    "assault_offenses",
    "robbery",
    "drug_narcotic_violations",
]


def map_crime_type(incident_type):
    if pd.isna(incident_type):
        return "others"
    nibrs_category = crime_type_mapping.get(incident_type, "others")

    process_string = lambda s: "_".join(
        [word.lower() for word in re.findall(r"\w+", s)]
    )
    processed = process_string(str(nibrs_category))
    return processed if processed in crime_types else "others"


def create_agent_features(
    df, lon_col, lat_col, census_col, race_col, i_lon_col="I_LON", i_lat_col="I_LAT"
):
    """Convert DataFrame rows to AgentFeatures objects"""
    agent_features_list = []

    # Convert coordinates to local projection once for efficiency
    coords_local = geopandas.GeoSeries.from_xy(
        x=df[lon_col], y=df[lat_col], crs=global_epsg
    ).to_crs(local_epsg)

    i_coords_local = geopandas.GeoSeries.from_xy(
        x=df[i_lon_col], y=df[i_lat_col], crs=global_epsg
    ).to_crs(local_epsg)

    for idx, (_, row) in enumerate(df.iterrows()):
        race_mapped = (
            race_remap.get(row[race_col], "NA") if pd.notna(row[race_col]) else "NA"
        )
        crime_type_mapped = map_crime_type(row["Type of Incident"])

        coord = coords_local.iloc[idx]
        home_coord = (float(coord.x), float(coord.y))

        coord = i_coords_local.iloc[idx]
        incident_coord = (float(coord.x), float(coord.y))

        home_block_id = (
            int(row[census_col])
            if pd.notna(row[census_col]) and row[census_col] != -1
            else None
        )
        assert home_block_id is not None

        incident_block_id = (
            int(row["I_CensusID"])
            if pd.notna(row["I_CensusID"]) and row["I_CensusID"] != -1
            else None
        )
        assert incident_block_id is not None

        agent_features = AgentFeatures(
            agent_id=idx,
            home_block_id=home_block_id,
            home_coord=home_coord,
            race=race_mapped,
            crime_type=crime_type_mapped,
            incident_block_id=incident_block_id,
            incident_block_coord=incident_coord,
        )

        agent_features_list.append(agent_features)

    return agent_features_list


def save_to_jsonl(agent_features_list, filepath):
    """Save list of AgentFeatures to JSONL file"""
    with open(filepath, "w") as f:
        for agent in agent_features_list:
            f.write(agent.model_dump_json() + "\n")


# Process offenders
print("Processing offenders...")
offenders_features = create_agent_features(
    df=offenders,
    lon_col="H_LON",
    lat_col="H_LAT",
    census_col="H_CensusID",
    race_col="Race",
)

# Process victims
print("Processing victims...")
victims_features = create_agent_features(
    df=victims,
    lon_col="V_LON",
    lat_col="V_LAT",
    census_col="V_CensusID",
    race_col="Victim Race",
)

if 'USE_DCM_PERIOD' in globals() and USE_DCM_PERIOD:
    output_dir = data_dir / "features/network"
    output_dir.mkdir(exist_ok=True, parents=True)
    offenders_path = output_dir / "offenders_2018_2020.jsonl"
    victims_path = output_dir / "victims_2018_2020.jsonl"
    print(f"💾 Saving DCM period data (2018-2020) to network directory")
else:
    offenders_path = data_dir / "features/biz/offenders_fine.jsonl"
    victims_path = data_dir / "features/biz/victims_fine.jsonl"
    print(f"Saving full period data to biz directory")

print(f"Saving {len(offenders_features)} offender records to {offenders_path}")
save_to_jsonl(offenders_features, offenders_path)

print(f"Saving {len(victims_features)} victim records to {victims_path}")
save_to_jsonl(victims_features, victims_path)

print("JSONL files created successfully")
print(f"   Offenders: {offenders_path}")
print(f"   Victims:   {victims_path}")